# 🚀 8K 480 FPS Style Viral TikTok Video Enhancer (Free T4 GPU)

Welcome! This Google Colab notebook provides **free NVIDIA T4 GPU** acceleration for:
- **Real-ESRGAN Neural Super-Resolution** (1080p -> 4K / 8K AI Upscale)
- **RIFE 4.x Deep Optical Flow** (Interpolate up to 60 / 120 / 480 FPS)
- **Viral TikTok CC & AMD CAS Edge Sharpening** (Deep S-curve contrast, specular highlight bloom, zero-artifact clarity)
- **Master Bitrate Export** (60-80 Mbps for that signature laggy crisp quality)

> **Step 1:** In Google Colab top menu, go to **Runtime > Change runtime type** and select **T4 GPU** (Free).

In [ ]:
#@title 1. Verify Free GPU Hardware & Clone Pipeline
!nvidia-smi

!git clone https://github.com/your-username/ULTRA-HIGH-RESOLUTION-TOOL.git /content/enhancer 2>/dev/null || true
%cd /content/enhancer
!pip install -q -r requirements.txt
!apt-get -y -qq update && apt-get -y -qq install ffmpeg

In [ ]:
#@title 2. Download Pre-Compiled NCNN Vulkan AI Binaries (RIFE + Real-ESRGAN)
import os

# Download RIFE NCNN Vulkan
if not os.path.exists('/usr/local/bin/rife-ncnn-vulkan'):
    print('[*] Installing RIFE AI frame interpolator...')
    !wget -q https://github.com/nihui/rife-ncnn-vulkan/releases/download/20221029/rife-ncnn-vulkan-20221029-ubuntu.zip
    !unzip -q -o rife-ncnn-vulkan-20221029-ubuntu.zip -d /tmp/rife
    !cp /tmp/rife/rife-ncnn-vulkan /usr/local/bin/ && chmod +x /usr/local/bin/rife-ncnn-vulkan

# Download Real-ESRGAN NCNN Vulkan
if not os.path.exists('/usr/local/bin/realesrgan-ncnn-vulkan'):
    print('[*] Installing Real-ESRGAN AI Super-Resolution...')
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesrgan-ncnn-vulkan-v0.2.5.0-ubuntu.zip
    !unzip -q -o realesrgan-ncnn-vulkan-v0.2.5.0-ubuntu.zip -d /tmp/realesrgan
    !cp /tmp/realesrgan/realesrgan-ncnn-vulkan /usr/local/bin/ && chmod +x /usr/local/bin/realesrgan-ncnn-vulkan

print('[+] AI Binaries installed and ready!')

In [ ]:
#@title 3. Video Processing Configuration & Execution
video_url = "" #@param {type:"string"}
preset = "viral_tiktok_hdr" #@param ["viral_tiktok_hdr", "velocity_flow_60fps", "alight_motion_dark", "cyberpunk_neon", "raw_master_8k"]
resolution = "4k" #@param ["1080p", "2k", "4k", "8k"]
target_fps = 60 #@param [30, 60, 120]
motion_mode = "blend" #@param ["blend", "mci", "framerate"]
codec = "h264" #@param ["h264", "h265"]
enable_bloom = True #@param {type:"boolean"}
sharpness_override = 0.8 #@param {type:"slider", min:0.0, max:1.0, step:0.05}

bloom_flag = "" if enable_bloom else "--no-bloom"

# If no URL provided, allow uploading from local PC
import os
input_path = video_url.strip()
if not input_path:
    from google.colab import files
    print("[*] Please upload your video:")
    uploaded = files.upload()
    input_path = list(uploaded.keys())[0]

output_path = f"/content/outputs/enhanced_{preset}_{resolution}.mp4"
os.makedirs("/content/outputs", exist_ok=True)

!python core/pipeline.py \
  -i "{input_path}" \
  -o "{output_path}" \
  -p "{preset}" \
  -r "{resolution}" \
  -fps {target_fps} \
  -m "{motion_mode}" \
  --codec "{codec}" \
  --sharpness {sharpness_override} \
  {bloom_flag}

In [ ]:
#@title 4. Download Processed Video to Your Phone / PC
from google.colab import files
import glob

files_to_download = glob.glob("/content/outputs/*.mp4")
if files_to_download:
    latest_file = max(files_to_download, key=os.path.getctime)
    print(f"[*] Downloading: {latest_file}")
    files.download(latest_file)
    
    # Also download side-by-side comparison image if exists
    comp_img = latest_file.replace('.mp4', '_comparison.jpg')
    if os.path.exists(comp_img):
        files.download(comp_img)
else:
    print("[ERROR] No output video found in /content/outputs/")